# Chapter 20: Working with Large Datasets: Chunking, Streaming, and Out-of-Core Processing

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [20]:
import pandas as pd
import numpy as np

# Working with Large Datasets: Chunking, Streaming, and Out-of-Core Processing

Real-world datasets often exceed available RAM. Rather than being limited by memory constraints, you can process arbitrarily large datasets by breaking them into manageable pieces. This chapter explores practical techniques for handling data that doesn't fit in memory using pandas and complementary tools.

## Understanding Memory Constraints

Before diving into solutions, it helps to understand the problem concretely. Let's measure how much memory a DataFrame actually consumes:

In [21]:
import pandas as pd
import numpy as np
import psutil

# Check your available memory
available_memory = psutil.virtual_memory().available / (1024**3)
print(f"Available RAM: {available_memory:.2f} GB")

# Estimate DataFrame size
df = pd.DataFrame({
    'id': np.arange(1_000_000),
    'value': np.random.randn(1_000_000),
    'category': np.random.choice(['A', 'B', 'C'], 1_000_000)
})

memory_usage = df.memory_usage(deep=True).sum() / (1024**2)
print(f"DataFrame size: {memory_usage:.2f} MB")

Available RAM: 16.97 GB
DataFrame size: 70.57 MB


This gives you a baseline. A million-row DataFrame with a few columns can easily consume hundreds of megabytes — and real datasets are often far larger. The strategies in this chapter address that problem systematically, starting with the simplest wins and progressing to more powerful tools.

## Strategy 1: Data Type Optimization

Before considering chunking or streaming, optimize your data types. This is the lowest-effort, highest-reward step and should always come first.

### Converting to Categorical Data

Categorical data is ideal for columns with repeated string values:

In [22]:
import pandas as pd
import numpy as np

# Before optimization
df = pd.DataFrame({
    'product': ['Widget'] * 500_000 + ['Gadget'] * 500_000,
    'region': np.random.choice(['North', 'South', 'East', 'West'], 1_000_000),
    'sales': np.random.randn(1_000_000)
})

print(f"Original memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# After optimization
df['product'] = df['product'].astype('category')
df['region'] = df['region'].astype('category')

print(f"Optimized memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Original memory: 126.36 MB
Optimized memory: 9.54 MB


### Downscaling Numeric Types

Use smaller numeric types when your data range allows it:

In [23]:
df = pd.DataFrame({
    'small_int': np.random.randint(0, 100, 1_000_000),
    'float_data': np.random.randn(1_000_000)
})

print(f"Before: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(df.dtypes)

# Downscale integer column (values 0-100 fit in int8)
df['small_int'] = df['small_int'].astype('int8')

# Downscale float column
df['float_data'] = df['float_data'].astype('float32')

print(f"\nAfter: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(df.dtypes)

Before: 15.26 MB
small_int       int64
float_data    float64
dtype: object

After: 4.77 MB
small_int        int8
float_data    float32
dtype: object


### Automating Dtype Optimization

When loading a CSV file, you can automate these optimizations:

In [24]:
import pandas as pd

def optimize_dtypes(df):
    """Reduce DataFrame memory footprint by optimizing column types."""
    original_memory = df.memory_usage(deep=True).sum()

    # Convert low-cardinality object columns to category
    for col in df.select_dtypes(include='object').columns:
        num_unique = df[col].nunique()
        num_total = len(df)
        if num_unique / num_total < 0.05:  # Less than 5% unique values
            df[col] = df[col].astype('category')

    # Downcast numeric columns
    for col in df.select_dtypes(include='integer').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    for col in df.select_dtypes(include='float').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')

    optimized_memory = df.memory_usage(deep=True).sum()
    reduction = 1 - (optimized_memory / original_memory)
    print(f"Memory reduction: {reduction:.1%}")
    return df

# Usage
df = pd.read_csv('data/large_file.csv')
df = optimize_dtypes(df)

Memory reduction: 97.0%


## Strategy 2: Chunking — Processing Files in Batches

Chunking is the right approach when you need to process multiple files independently, or when a single file is too large to load at once. The key principle: each chunk fits in memory, but the total dataset does not.

### Reading a Large CSV File in Chunks

In [25]:
import pandas as pd

chunk_size = 50_000
chunks = []

for chunk in pd.read_csv('data/large_file.csv', chunksize=chunk_size):
    # Process each chunk independently
    chunk['Age'] = chunk['Age'].astype('float32')
    chunk = chunk[chunk['Age'] > chunk['Age'].quantile(0.25)]
    chunks.append(chunk)

# Combine results — only do this if the final result fits in memory
result = pd.concat(chunks, ignore_index=True)
print(f"Result shape: {result.shape}")

Result shape: (1480, 3)


### Batch Processing Multiple Files

In [ ]:
import os, glob
import pathlib
import pandas as pd

# Generate sample log CSV files
os.makedirs('data/logs', exist_ok=True)
for i in range(3):
    pd.DataFrame({'timestamp': pd.date_range('2024-01-01', periods=100), 'value': range(100)}).to_csv(f'data/logs/log_{i}.csv', index=False)

# Process all CSV files in a directory
data_dir = pathlib.Path('data/logs/')
all_results = []

for csv_file in sorted(data_dir.glob('*.csv')):
    df = pd.read_csv(csv_file)
    # Perform analysis on this file
    summary = df.groupby('user_id')['action'].count() if 'user_id' in df.columns else df['value'].sum()
    all_results.append(summary)

# Combine per-file summaries — these are small, so this is safe
if all_results:
    if isinstance(all_results[0], pd.Series):
        combined = pd.concat(all_results).groupby(level=0).sum()
        print(combined.head())
    else:
        print(f"Total value across files: {sum(all_results)}")

### Converting CSV Files to Parquet

Parquet is a columnar format that is far more efficient than CSV for large datasets. Converting once pays dividends on every subsequent read:

In [ ]:
import pathlib
import pandas as pd

data_dir = pathlib.Path('data/raw_logs/')
output_dir = pathlib.Path('data/processed/')
output_dir.mkdir(parents=True, exist_ok=True)

for csv_file in sorted(data_dir.glob('*.csv')):
    print(f"Processing {csv_file.name}...")

    df = pd.read_csv(csv_file)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df[df['value'] > 0]

    output_file = output_dir / f"{csv_file.stem}.parquet"
    df.to_parquet(output_file, compression='snappy')

print("Conversion complete!")

### A Reusable Chunking Function

In [ ]:
import pandas as pd

def process_in_chunks(file_path, chunk_size=50_000, process_func=None):
    """
    Process a large CSV file in chunks.

    Parameters
    ----------
    file_path : str
        Path to the CSV file.
    chunk_size : int
        Number of rows per chunk.
    process_func : callable, optional
        Function to apply to each chunk. Must return a DataFrame.

    Returns
    -------
    pd.DataFrame or None
    """
    results = []

    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        if process_func:
            result = process_func(chunk)
            results.append(result)

    return pd.concat(results, ignore_index=True) if results else None

# Usage
def my_processor(chunk):
    chunk['new_col'] = chunk['value'] * 2
    return chunk[chunk['new_col'] > 100]

result = process_in_chunks('data.csv', chunk_size=50_000, process_func=my_processor)

## Strategy 3: Out-of-Core Aggregations

When you only need a summary — a count, a sum, a mean — you don't need to hold all the data in memory at once. Maintain running totals instead.

### Counting Values Across Multiple Files

In [ ]:
import pathlib
import pandas as pd

data_dir = pathlib.Path('data/timeseries/')
counts = pd.Series(dtype='int64')

for parquet_file in sorted(data_dir.glob('ts-*.parquet')):
    print(f"Processing {parquet_file.name}...")
    df = pd.read_parquet(parquet_file)

    file_counts = df['category'].value_counts()
    counts = counts.add(file_counts, fill_value=0)

counts = counts.astype('int64')
print(counts.sort_values(ascending=False).head(10))

### Computing a Mean Across Multiple Files

In [ ]:
import pathlib
import pandas as pd

total_sum = 0
total_count = 0

for file_path in sorted(pathlib.Path('data/').glob('*.csv')):
    df = pd.read_csv(file_path)
    total_sum += df['value'].sum()
    total_count += len(df)

mean_value = total_sum / total_count
print(f"Mean across all files: {mean_value:.4f}")

### Streaming GroupBy Aggregation

In [ ]:
import pathlib
import pandas as pd

aggregated = None

for file_path in sorted(pathlib.Path('data/').glob('*.csv')):
    df = pd.read_csv(file_path)
    file_agg = df.groupby('category')['amount'].sum()

    if aggregated is None:
        aggregated = file_agg
    else:
        aggregated = aggregated.add(file_agg, fill_value=0)

print(aggregated)

## Strategy 4: Streaming Statistics

Some statistics — like the mean and standard deviation — can be computed incrementally using online algorithms. This avoids storing any intermediate data.

In [ ]:
import pandas as pd
import numpy as np

# Generate sample large_file.csv for this example
pd.DataFrame({'value': range(10000)}).to_csv('large_file.csv', index=False)

class StreamingStats:
    """Calculate mean and standard deviation without loading the entire dataset."""

    def __init__(self):
        self.count = 0
        self.mean = 0.0
        self.M2 = 0.0  # Accumulator for variance (Welford's algorithm)

    def update(self, values):
        """Update running statistics with a new batch of values."""
        for value in values:
            self.count += 1
            delta = value - self.mean
            self.mean += delta / self.count
            delta2 = value - self.mean
            self.M2 += delta * delta2

    def finalize(self):
        """Return final statistics."""
        variance = self.M2 / (self.count - 1) if self.count > 1 else 0.0
        return {
            'count': self.count,
            'mean': self.mean,
            'std': np.sqrt(variance)
        }

# Usage
stats = StreamingStats()

for chunk in pd.read_csv('large_file.csv', chunksize=10_000):
    stats.update(chunk['value'].dropna().values)

results = stats.finalize()
print(f"Count: {results['count']}")
print(f"Mean:  {results['mean']:.4f}")
print(f"Std:   {results['std']:.4f}")

Welford's algorithm is numerically stable and processes each value exactly once, making it well-suited for streaming scenarios.

## Strategy 5: Sparse Data Structures

When your data contains many missing or repeated values, sparse arrays store only the non-fill values, which can yield dramatic memory savings.

In [ ]:
import pandas as pd
import numpy as np

# Simulate a mostly-NaN array
dense_array = np.random.randn(1_000_000)
dense_array[100:999_900] = np.nan  # 99.98% of values are NaN

# Regular Series
regular_series = pd.Series(dense_array)
dense_memory = regular_series.memory_usage(deep=True) / (1024**2)

# Sparse Series
sparse_series = pd.Series(pd.arrays.SparseArray(dense_array))
sparse_memory = sparse_series.memory_usage(deep=True) / (1024**2)

print(f"Dense:  {dense_memory:.2f} MB")
print(f"Sparse: {sparse_memory:.2f} MB")
print(f"Compression ratio: {dense_memory / sparse_memory:.1f}x")

Sparse arrays are most effective when a large fraction of values are identical (typically NaN or zero). For dense data, the overhead of the sparse representation can actually increase memory usage.

## Strategy 6: Using Dask for Parallel Processing

For complex workflows involving many files or multi-step transformations, Dask provides a pandas-compatible API that executes operations in parallel and out-of-core:

In [ ]:
import dask.dataframe as dd

# Read multiple parquet files as a single Dask DataFrame
ddf = dd.read_parquet('data/timeseries/*.parquet')

# Operations are lazy — nothing executes until .compute() is called
result = ddf.groupby('category')['value'].mean().compute()
print(result)

# More complex example: filter, then aggregate
result = (ddf
    .query('value > 0')
    .groupby('category')
    .agg({'value': ['mean', 'std', 'count']})
    .compute())

print(result)

Dask is a natural next step once the manual chunking patterns in earlier strategies become unwieldy. It handles task scheduling, parallelism, and memory management automatically.

## Practical Workflow: Putting It All Together

Here is a complete workflow that combines dtype optimization, chunked reading, and Parquet output:

In [ ]:
import pathlib
import pandas as pd

def process_large_dataset(input_dir: str, output_file: str, chunk_size: int = 50_000):
    """
    Read multiple CSV files in chunks, optimize dtypes, and save to Parquet.

    Parameters
    ----------
    input_dir : str
        Directory containing input CSV files.
    output_file : str
        Path for the output Parquet file.
    chunk_size : int
        Rows per chunk when reading each CSV.

    Returns
    -------
    pd.DataFrame
        The combined, optimized DataFrame.
    """
    input_path = pathlib.Path(input_dir)
    all_chunks = []

    # Step 1: Read and optimize each file in chunks
    for csv_file in sorted(input_path.glob('*.csv')):
        print(f"Reading {csv_file.name}...")

        for chunk in pd.read_csv(csv_file, chunksize=chunk_size):
            # Optimize dtypes
            chunk['date'] = pd.to_datetime(chunk['date'])
            chunk['category'] = chunk['category'].astype('category')
            chunk['value'] = pd.to_numeric(chunk['value'], downcast='float')

            # Drop rows with missing required values
            chunk = chunk.dropna(subset=['value'])

            all_chunks.append(chunk)

    # Step 2: Combine all chunks
    print("Combining chunks...")
    result = pd.concat(all_chunks, ignore_index=True)

    # Step 3: Save in an efficient format
    print(f"Saving to {output_file}...")
    result.to_parquet(output_file, compression='snappy')

    return result

# Usage
df = process_large_dataset('data/raw/', 'data/processed.parquet')
print(f"Final shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## Practical Example: Analyzing Web Logs

Here is a realistic end-to-end example that applies streaming aggregation to web server logs spread across multiple files:

In [ ]:
import pathlib
import pandas as pd

def analyze_logs(log_dir: str, chunk_size: int = 50_000):
    """
    Analyze web logs from multiple files without loading all data into memory.

    Returns daily request counts and HTTP status code distributions.
    """
    log_dir = pathlib.Path(log_dir)

    daily_stats = pd.Series(dtype='int64')
    status_counts = pd.Series(dtype='int64')

    for log_file in sorted(log_dir.glob('*.log')):
        print(f"Processing {log_file.name}...")

        for chunk in pd.read_csv(log_file, chunksize=chunk_size, sep=r'\s+'):
            # Extract date from timestamp
            chunk['date'] = pd.to_datetime(chunk['timestamp']).dt.date

            # Accumulate daily request counts
            daily = chunk.groupby('date').size()
            daily_stats = daily_stats.add(daily, fill_value=0)

            # Accumulate status code counts
            status = chunk['status'].value_counts()
            status_counts = status_counts.add(status, fill_value=0)

    return daily_stats.astype('int64'), status_counts.astype('int64')

# Usage
# daily, statuses = analyze_logs('logs/')
# print(daily.sort_index().tail(7))
# print(statuses.sort_values(ascending=False))

## Key Takeaways

| Strategy | Best For | Pros | Cons |
|---|---|---|---|
| **Type Optimization** | All datasets | Simple, immediate gains | Limited reduction ceiling |
| **Chunking** | Independent per-chunk operations | Simple, predictable | Requires custom aggregation logic |
| **Streaming Aggregation** | Counts, sums, means | Very memory efficient | Limited to associative operations |
| **Streaming Statistics** | Mean, std over large data | Numerically stable, single pass | Custom implementation required |
| **Sparse Arrays** | Mostly-NaN or mostly-zero data | Extreme memory savings | Only effective for sparse patterns |
| **Dask** | Complex multi-step workflows | Parallel, scalable, pandas-like API | Additional dependency |

A few guiding principles:

- **Optimize data types first.** Converting strings to categories and downcasting numerics is free and often cuts memory use by half or more.
- **Prefer Parquet over CSV** for any file you will read more than once. Parquet is columnar, compressed, and preserves dtypes.
- **Stream aggregations rather than accumulating chunks** whenever your final result is a summary rather than a full table.
- **Combine strategies.** Type optimization + chunked reads + Parquet output is a powerful default workflow for most large-dataset problems.
- **Reach for Dask** when manual chunking becomes complex or when you want to exploit multiple CPU cores.

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Optimize DataFrame Memory with Dtype Casting

Given a DataFrame with inefficient data types, reduce its memory footprint by downcasting numeric columns and converting low-cardinality string columns to the 'category' dtype. Print the memory usage before and after optimization.

In [ ]:
import pandas as pd
import numpy as np

# Sample sales DataFrame with inefficient dtypes
df = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005],
    'quantity': [2, 5, 1, 8, 3],
    'price': [19.99, 5.49, 99.95, 3.25, 45.00],
    'region': ['North', 'South', 'North', 'East', 'South'],
    'status': ['shipped', 'pending', 'shipped', 'shipped', 'pending']
})

print("=== Before Optimization ===")
print(df.dtypes)
print(f"Memory usage: {df.memory_usage(deep=True).sum()} bytes\n")

# TODO: Downcast 'order_id' to the smallest suitable unsigned integer type

# TODO: Downcast 'quantity' to the smallest suitable unsigned integer type

# TODO: Downcast 'price' to float32

# TODO: Convert 'region' and 'status' columns to 'category' dtype

print("=== After Optimization ===")
print(df.dtypes)
print(f"Memory usage: {df.memory_usage(deep=True).sum()} bytes")


### Exercise 2: Process a Large CSV in Chunks

Simulate chunked processing of a large dataset by writing a CSV to disk and reading it back in chunks of a fixed size. Compute the total revenue (quantity * price) across all chunks without loading the entire file into memory at once.

In [ ]:
import pandas as pd
import numpy as np
import io

# Simulate a large CSV by creating and saving sample data
np.random.seed(42)
n_rows = 1000
data = pd.DataFrame({
    'order_id': range(1, n_rows + 1),
    'quantity': np.random.randint(1, 20, size=n_rows),
    'price': np.round(np.random.uniform(5.0, 200.0, size=n_rows), 2),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food'], size=n_rows)
})
data.to_csv('orders.csv', index=False)

# TODO: Set a chunk size of 200 rows
chunk_size = None

# TODO: Initialize a variable to accumulate total revenue
total_revenue = None

# TODO: Use pd.read_csv with the chunksize parameter to iterate over chunks.
#       For each chunk, compute revenue (quantity * price) and add to total_revenue.


print(f"Total revenue across all chunks: ${total_revenue:,.2f}")

# Verify against full-file calculation
full_revenue = (data['quantity'] * data['price']).sum()
print(f"Verification (full load):         ${full_revenue:,.2f}")


### Exercise 3: Out-of-Core Group Aggregation with Chunking

Using chunked reading, compute the total quantity sold per product category from a CSV file without ever loading the full dataset. Accumulate partial group results across chunks and combine them into a final summary.

In [ ]:
import pandas as pd
import numpy as np

# Create and save sample data
np.random.seed(7)
n_rows = 500
df_full = pd.DataFrame({
    'transaction_id': range(1, n_rows + 1),
    'category': np.random.choice(['Books', 'Toys', 'Sports', 'Garden'], size=n_rows),
    'quantity': np.random.randint(1, 15, size=n_rows)
})
df_full.to_csv('transactions.csv', index=False)

# TODO: Create an empty Series to accumulate quantity totals per category
category_totals = None

# TODO: Read 'transactions.csv' in chunks of 100 rows.
#       For each chunk, group by 'category', sum 'quantity',
#       and add the result to category_totals.
#       Hint: Use the .add() method with fill_value=0 to combine Series.


print("Total quantity sold per category (chunked):")
print(category_totals.sort_index())

# Verify
verification = df_full.groupby('category')['quantity'].sum()
print("\nVerification (full load):")
print(verification.sort_index())


### Exercise 4: Streaming Mean and Variance with Welford's Algorithm

Implement a streaming statistics tracker that computes the running mean and variance of a numeric column one chunk at a time, without storing all values in memory. Use Welford's online algorithm to update the mean and sum of squared deviations incrementally across chunks.

In [ ]:
import pandas as pd
import numpy as np

# Generate and save a dataset
np.random.seed(0)
n_rows = 800
df_full = pd.DataFrame({
    'sensor_id': np.random.choice(['S1', 'S2', 'S3'], size=n_rows),
    'reading': np.random.normal(loc=50.0, scale=10.0, size=n_rows)
})
df_full.to_csv('sensor_data.csv', index=False)

# Welford's online algorithm accumulators
count = 0       # total number of values seen
mean = 0.0      # running mean
M2 = 0.0        # running sum of squared deviations from the mean

# TODO: Read 'sensor_data.csv' in chunks of 150 rows.
#       For each value in the 'reading' column of each chunk,
#       update count, mean, and M2 using Welford's algorithm:
#           count += 1
#           delta = value - mean
#           mean += delta / count
#           delta2 = value - mean
#           M2 += delta * delta2


# TODO: Compute the final variance as M2 / count
#       and standard deviation as its square root.
streaming_variance = None
streaming_std = None

print(f"Streaming mean:     {mean:.4f}")
print(f"Streaming variance: {streaming_variance:.4f}")
print(f"Streaming std dev:  {streaming_std:.4f}")

# Verify against numpy
print(f"\nNumPy mean:         {df_full['reading'].mean():.4f}")
print(f"NumPy variance:     {df_full['reading'].var(ddof=0):.4f}")
print(f"NumPy std dev:      {df_full['reading'].std(ddof=0):.4f}")


---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Optimize DataFrame Memory with Dtype Casting

In [ ]:
import pandas as pd
import numpy as np

# Sample sales DataFrame with inefficient dtypes
df = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005],
    'quantity': [2, 5, 1, 8, 3],
    'price': [19.99, 5.49, 99.95, 3.25, 45.00],
    'region': ['North', 'South', 'North', 'East', 'South'],
    'status': ['shipped', 'pending', 'shipped', 'shipped', 'pending']
})

print("=== Before Optimization ===")
print(df.dtypes)
print(f"Memory usage: {df.memory_usage(deep=True).sum()} bytes\n")

# Downcast 'order_id' to the smallest suitable unsigned integer type
df['order_id'] = pd.to_numeric(df['order_id'], downcast='unsigned')

# Downcast 'quantity' to the smallest suitable unsigned integer type
df['quantity'] = pd.to_numeric(df['quantity'], downcast='unsigned')

# Downcast 'price' to float32
df['price'] = df['price'].astype('float32')

# Convert 'region' and 'status' columns to 'category' dtype
df['region'] = df['region'].astype('category')
df['status'] = df['status'].astype('category')

print("=== After Optimization ===")
print(df.dtypes)
print(f"Memory usage: {df.memory_usage(deep=True).sum()} bytes")


### Solution 2: Process a Large CSV in Chunks

In [ ]:
import pandas as pd
import numpy as np

# Simulate a large CSV by creating and saving sample data
np.random.seed(42)
n_rows = 1000
data = pd.DataFrame({
    'order_id': range(1, n_rows + 1),
    'quantity': np.random.randint(1, 20, size=n_rows),
    'price': np.round(np.random.uniform(5.0, 200.0, size=n_rows), 2),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food'], size=n_rows)
})
data.to_csv('orders.csv', index=False)

# Set a chunk size of 200 rows
chunk_size = 200

# Initialize a variable to accumulate total revenue
total_revenue = 0.0

# Use pd.read_csv with the chunksize parameter to iterate over chunks
for chunk in pd.read_csv('orders.csv', chunksize=chunk_size):
    chunk_revenue = (chunk['quantity'] * chunk['price']).sum()
    total_revenue += chunk_revenue

print(f"Total revenue across all chunks: ${total_revenue:,.2f}")

# Verify against full-file calculation
full_revenue = (data['quantity'] * data['price']).sum()
print(f"Verification (full load):         ${full_revenue:,.2f}")


### Solution 3: Out-of-Core Group Aggregation with Chunking

In [ ]:
import pandas as pd
import numpy as np

# Create and save sample data
np.random.seed(7)
n_rows = 500
df_full = pd.DataFrame({
    'transaction_id': range(1, n_rows + 1),
    'category': np.random.choice(['Books', 'Toys', 'Sports', 'Garden'], size=n_rows),
    'quantity': np.random.randint(1, 15, size=n_rows)
})
df_full.to_csv('transactions.csv', index=False)

# Create an empty Series to accumulate quantity totals per category
category_totals = pd.Series(dtype='float64')

# Read 'transactions.csv' in chunks of 100 rows
for chunk in pd.read_csv('transactions.csv', chunksize=100):
    chunk_totals = chunk.groupby('category')['quantity'].sum()
    category_totals = category_totals.add(chunk_totals, fill_value=0)

print("Total quantity sold per category (chunked):")
print(category_totals.sort_index())

# Verify
verification = df_full.groupby('category')['quantity'].sum()
print("\nVerification (full load):")
print(verification.sort_index())


### Solution 4: Streaming Mean and Variance with Welford's Algorithm

In [ ]:
import pandas as pd
import numpy as np

# Generate and save a dataset
np.random.seed(0)
n_rows = 800
df_full = pd.DataFrame({
    'sensor_id': np.random.choice(['S1', 'S2', 'S3'], size=n_rows),
    'reading': np.random.normal(loc=50.0, scale=10.0, size=n_rows)
})
df_full.to_csv('sensor_data.csv', index=False)

# Welford's online algorithm accumulators
count = 0
mean = 0.0
M2 = 0.0

# Read 'sensor_data.csv' in chunks of 150 rows
for chunk in pd.read_csv('sensor_data.csv', chunksize=150):
    for value in chunk['reading']:
        count += 1
        delta = value - mean
        mean += delta / count
        delta2 = value - mean
        M2 += delta * delta2

# Compute the final variance and standard deviation
streaming_variance = M2 / count
streaming_std = streaming_variance ** 0.5

print(f"Streaming mean:     {mean:.4f}")
print(f"Streaming variance: {streaming_variance:.4f}")
print(f"Streaming std dev:  {streaming_std:.4f}")

# Verify against numpy
print(f"\nNumPy mean:         {df_full['reading'].mean():.4f}")
print(f"NumPy variance:     {df_full['reading'].var(ddof=0):.4f}")
print(f"NumPy std dev:      {df_full['reading'].std(ddof=0):.4f}")
